# (Re)projecting Spatial Data


In the previous section, we covered the basics of map projections and coordinate reference systems.

In this section, we will look at how to:

- choose an appropriate projection for spatial analysis;
- reproject data into a different coordinate reference system;
- assign a CRS when one is missing.


## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import geopandas as gpd
import pandas as pd
import osmnx as ox

# cache OSM responses on disk, so repeating a query does not hit the server again
ox.settings.cache_folder = "../../cache"


### 0.2. Preparing the Data


Start by loading the district boundary from OpenStreetMap.


In [ ]:
area_name = "Innere Stadt, Vienna, Austria"

admin_border = ox.geocode_to_gdf(area_name)
admin_border.explore(tiles="cartodbpositron")

## 1. Checking the Coordinate Reference System


When working with spatial data, one of the first steps is to check the coordinate reference system the data is in. This tells you how to interpret the coordinates and whether the data is suitable for further spatial analysis.

Before you start, check:

- whether a **CRS** is defined for the data;
- what **type of CRS** is being used;
- whether the **units of measurement** are appropriate for the analysis;
- whether all datasets share the **same CRS**.


Check the CRS of the data:


In [ ]:
admin_border.crs

Our data is in the geographic coordinate system **WGS 84 (EPSG:4326)**.
Feature coordinates are stored as **degrees of latitude and longitude**.


## 2. Identifying the Appropriate UTM Zone


When working with cities or local areas, **UTM** is one of the most commonly used coordinate systems.

As mentioned in the previous section, data in a **geographic coordinate system** is measured in **degrees**, and the ground distance represented by one degree of longitude **varies with latitude**. This makes geographic coordinates unsuitable for accurate measurement of **distances and areas**.

The **UTM (Universal Transverse Mercator)** coordinate system uses **metres**, which allows distances, areas, and directions to be calculated much more accurately when working at a local scale.

So how do you find the right UTM zone for a dataset?


### 2.1. Using a Formula

The UTM zone can be calculated directly from a point's coordinates.

Only the **longitude** is needed to determine the **zone number**.
The **latitude** is used to determine whether the point lies in the **northern** or **southern hemisphere**.

The UTM zone number can be calculated using the following formula:

$$
\text{zone} = \left\lfloor \frac{\text{longitude} + 180}{6} \right\rfloor + 1
$$

First, the longitude is shifted from the range `[-180, +180]` to `[0, 360]`. The result is then divided by 6, since each UTM zone spans 6° of longitude. The value is floored to get the zone index, and 1 is added because UTM zones are numbered starting from 1, not 0.

Once the zone number is known, the corresponding **EPSG code** can be derived:

- **EPSG:326xx** — for zones in the **northern hemisphere**
- **EPSG:327xx** — for zones in the **southern hemisphere**

where `xx` is the UTM zone number.


Let's write a small function that takes a longitude and latitude and returns the corresponding UTM EPSG code.


In [ ]:
def utm_epsg(longitude, latitude):
    zone = int((longitude + 180) // 6) + 1
    zone = min(zone, 60)  # longitude 180° falls into the last zone

    if latitude >= 0:
        epsg = 32600 + zone   # northern hemisphere
    else:
        epsg = 32700 + zone   # southern hemisphere

    return epsg

Try it with the coordinates of Vienna:


In [ ]:
lon = 16.37
lat = 48.21

utm_epsg(lon, lat)

This EPSG code can be used directly to reproject the data in the steps that follow.


### 2.2. Automatic Detection

In practice, calculating the UTM zone manually is rarely necessary.
GeoPandas can **automatically determine the appropriate UTM CRS** using the `.estimate_utm_crs()` method.

This method examines the **geographic extent of the data** and selects the UTM zone based on the centroid of the layer's geometry.

In [ ]:
utm_crs = admin_border.estimate_utm_crs()

print(f"EPSG code: {utm_crs.to_epsg()}")

utm_crs

The EPSG code matches the one we obtained earlier using the formula (**32633** — UTM zone 33N). It can be used directly to reproject the data.

## 3. Reprojecting

**Reprojection** is the process of converting spatial data from one coordinate reference system to another.

In practice, this means **recalculating the coordinates of every point** so that they correspond to a different map projection.

First, **how the coordinates are stored right now in the `geometry` column**.


In [ ]:
admin_border.geometry

We can see that each vertex of the polygon is represented as a coordinate pair. In the geographic CRS EPSG:4326, these values correspond to longitude and latitude, expressed in degrees.


To reproject data in **GeoPandas**, we use the `.to_crs()` method.

This method **recalculates the coordinates of all geometries** so that the data is correctly positioned in the new coordinate system.

Let's reproject the data into the appropriate **UTM zone** that we identified earlier using `estimate_utm_crs()` and stored in the `utm_crs` variable.

In [ ]:
admin_border_utm = admin_border.to_crs(utm_crs)

First, let's confirm that the coordinate system has actually changed.


In [ ]:
admin_border_utm.crs

The data is now in a projected UTM coordinate system, where coordinates are expressed in metres.

And here is what the coordinates look like now.


In [ ]:
admin_border_utm.geometry

Unlike before, the coordinates are now in metres rather than degrees. This is evident from the order of magnitude: the values are now much larger — typically in the hundreds of thousands or millions.


> We have successfully reprojected the layer into the appropriate **UTM zone**.
> The data is now in a coordinate system suitable for **accurate spatial calculations** — such as measuring distances, computing areas, and performing other analytical operations.


## 4. Assigning a CRS

Spatial data sometimes **does not include CRS information**.
In that case, GeoPandas **cannot correctly interpret the geometry coordinates**, because it does not know which coordinate system they are in.


### 4.1. Data Without a Defined CRS

When spatial data has **no CRS defined**, you need to **explicitly assign one** so that the coordinates can be interpreted correctly.

Here is an example with a **CSV file** of well-known places to visit in Vienna.

The file is **vienna_top_locations.csv** from `data/vienna/`. _Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._

We read the data and build a `GeoDataFrame` from it. As in the first module, we declare the semicolon separator and the comma decimal mark, then drop the rows that have no coordinates:

In [ ]:
df_csv = pd.read_csv("../../data/vienna/vienna_top_locations.csv", sep=";", decimal=",")
df_csv = df_csv.dropna(subset=["geo_longitude", "geo_latitude"])

gdf_csv = gpd.GeoDataFrame(
    df_csv,
    geometry=gpd.points_from_xy(df_csv["geo_longitude"], df_csv["geo_latitude"])
)

Which CRS does `gdf_csv` have?


In [ ]:
print(f"CRS: {gdf_csv.crs}")

The result is `None`, because we **did not specify a CRS** when creating the `GeoDataFrame`.

In practice, this means GeoPandas **has no way of knowing how to interpret the geometry coordinates**.

In our case, the coordinates are **longitude and latitude**, so we can reasonably assume they are in the geographic coordinate system **WGS 84 (EPSG:4326)**.

If no CRS is set, it can be added in two ways:

- **Option 1:** use the `.set_crs()` method
- **Option 2:** assign a value directly via the `.crs` attribute (shown commented out below — the two options do exactly the same thing)

(It is worth noting that the best practice is to **specify the CRS when creating the `GeoDataFrame`**. This avoids potential errors in subsequent spatial operations.)

In [ ]:
gdf_csv = gdf_csv.set_crs(epsg=4326)  # option 1

# gdf_csv.crs = "EPSG:4326"           # option 2 — the same result

print(f"CRS: {gdf_csv.crs}")

Assigning a CRS **does not change the coordinates** — it simply **tells GeoPandas which coordinate system they are in**.


> **The key distinction when working with CRS**
>
> **Reprojecting (`to_crs`)**
>
> Converts coordinates from one CRS to another.
> Both the **CRS and the coordinate values of all geometries** are changed.
>
> **Assigning a CRS (`set_crs`)**
>
> Specifies the coordinate system without modifying the coordinates.
> You are simply **telling GeoPandas how to interpret the existing coordinates**.


## Summary


In this section, we covered how to **reproject spatial data** and how to assign a coordinate reference system when one is missing.

We learned how to:

- identify the appropriate **UTM zone** for a dataset using a built-in method;
- use `.to_crs()` to **reproject data**;
- distinguish between **assigning a CRS** (telling GeoPandas which system the coordinates are in, without changing them) and **reprojecting** (actually transforming the coordinate values).

Working correctly with coordinate reference systems is an essential part of preparing spatial data for analysis.
